In [ ]:

import os
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.ops import DeformConv2d


INPUT_DIR = "data"
OUTPUT_MASK_DIR = "output_mask"

PATIENT_FOLDERS = [
    "1",
    "1 (32)",
    "1 (39)",
]
CHECKPOINT_PATH = "best_model.pth"

IMAGE_SIZE = (256, 256) 
THRESHOLD = 0.5
DROPOUT_RATE = 0.3
BATCH_SIZE = 1
RECURSIVE = True
RESTORE_ORIGINAL_SIZE = True
SKIP_MASK_NAMED_FILES = True

NORMALIZATION_MODE = "fixed"
TRAIN_MEAN = 0.3081
TRAIN_STD = 0.2153

SUPPORTED_EXTENSIONS = {".jpg"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if hasattr(Image, "Resampling"):
    BICUBIC = Image.Resampling.BICUBIC
    NEAREST = Image.Resampling.NEAREST
else:
    BICUBIC = Image.BICUBIC
    NEAREST = Image.NEAREST


class CoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super(CoordinateAttention, self).__init__()

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mid_channels = max(8, in_channels // reduction)

        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(mid_channels, in_channels, kernel_size=1)
        self.conv_w = nn.Conv2d(mid_channels, in_channels, kernel_size=1)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)

        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        att_h = self.sigmoid(self.conv_h(x_h))
        att_w = self.sigmoid(self.conv_w(x_w))

        return identity * att_h * att_w


class DualPathCBAM(nn.Module):
    def __init__(self, channels):
        super(DualPathCBAM, self).__init__()

        self.ca = CoordinateAttention(channels)

        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

        self.conv = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        x_att = self.ca(x)

        max_pool = torch.max(x_att, dim=1, keepdim=True)[0]
        avg_pool = torch.mean(x_att, dim=1, keepdim=True)

        spatial = torch.cat([max_pool, avg_pool], dim=1)

        x_att = x_att * self.sa(spatial)
        x_res = self.conv(x)

        return x_att + x_res


class LearnableUpsample(nn.Module):
    def __init__(self, in_channels, upscale_factor):
        super(LearnableUpsample, self).__init__()

        self.upsample = nn.ConvTranspose2d(
            in_channels,
            in_channels,
            kernel_size=upscale_factor * 2 - (upscale_factor % 2),
            stride=upscale_factor,
            padding=upscale_factor // 2
        )

    def forward(self, x):
        return self.upsample(x)


class DenseResidualBlockWithCrossLayerConnections(nn.Module):
    def __init__(self, in_c, out_c, dropout_rate=0.3):
        super().__init__()

        self.gelu = nn.GELU()

        self.conv_offset1 = nn.Conv2d(
            in_c,
            18,
            kernel_size=3,
            padding=1
        )

        self.conv1 = DeformConv2d(
            in_c,
            out_c,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(out_c)
        self.dropout1 = nn.Dropout2d(dropout_rate)

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=1),
            nn.BatchNorm2d(out_c)
        )

    def forward(self, inputs):
        offset1 = self.conv_offset1(inputs)

        x1 = self.conv1(inputs, offset1)
        x1 = self.bn1(x1)
        x1 = self.gelu(x1)
        x1 = self.dropout1(x1)

        return x1 + self.shortcut(inputs)


class Bottleneck(nn.Module):
    def __init__(
        self,
        in_c,
        out_c,
        dim,
        num_layers=6,
        dropout_rate=0.3
    ):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=1),
            nn.BatchNorm2d(out_c),
            nn.GELU(),
            nn.Dropout2d(dropout_rate)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=8,
            batch_first=True
        )

        self.tblock = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.GELU(),
            nn.Dropout2d(dropout_rate)
        )

    def forward(self, x):
        x = self.conv1(x)

        b, c, h, w = x.shape

        x_seq = x.flatten(2).transpose(1, 2)
        x_seq = self.tblock(x_seq)

        x = x_seq.transpose(1, 2).contiguous().view(b, c, h, w)

        return self.conv2(x)


class DilatedConv(nn.Module):
    def __init__(self, in_c, out_c, dropout_rate=0.3):
        super().__init__()

        self.c1 = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.01),
            nn.Dropout2d(dropout_rate)
        )

        self.c2 = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.01),
            nn.Dropout2d(dropout_rate)
        )

        self.c3 = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=3, dilation=3),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.01),
            nn.Dropout2d(dropout_rate)
        )

        self.c4 = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=6, dilation=6),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.01),
            nn.Dropout2d(dropout_rate)
        )

        self.c5 = nn.Sequential(
            nn.Conv2d(out_c * 4, out_c, kernel_size=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.01),
            nn.Dropout2d(dropout_rate)
        )

    def forward(self, inputs):
        x1 = self.c1(inputs)
        x2 = self.c2(inputs)
        x3 = self.c3(inputs)
        x4 = self.c4(inputs)

        return self.c5(torch.cat([x1, x2, x3, x4], dim=1))


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_c,
        out_c,
        upscale_factor=2,
        dropout_rate=0.3
    ):
        super().__init__()

        self.learnable_upsample = LearnableUpsample(
            in_c[0],
            upscale_factor
        )

        self.conv_skip = nn.Conv2d(
            in_c[1],
            out_c,
            kernel_size=1
        )

        self.dual_path_cbam = DualPathCBAM(out_c)

        self.dense_res1 = DenseResidualBlockWithCrossLayerConnections(
            in_c[0] + out_c,
            out_c,
            dropout_rate=dropout_rate
        )

        self.dense_res2 = DenseResidualBlockWithCrossLayerConnections(
            out_c,
            out_c,
            dropout_rate=dropout_rate
        )

        self.dropout = nn.Dropout2d(dropout_rate)

    def forward(self, inputs, skip):
        x = self.learnable_upsample(inputs)

        skip = self.conv_skip(skip)
        skip = self.dual_path_cbam(skip)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        x = torch.cat([x, skip], dim=1)

        x = self.dense_res1(x)
        x = self.dense_res2(x)
        x = self.dropout(x)

        return x


class RCSDT_UNet(nn.Module):
    def __init__(
        self,
        dropout_rate=0.3,
        pretrained=False
    ):
        super().__init__()

        if pretrained:
            weights = models.ResNeXt101_64X4D_Weights.IMAGENET1K_V1
        else:
            weights = None

        backbone = models.resnext101_64x4d(weights=weights)

        self.layer0 = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            nn.GELU()
        )

        self.layer1 = nn.Sequential(
            backbone.maxpool,
            backbone.layer1
        )

        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3

        self.b1 = Bottleneck(
            in_c=1024,
            out_c=256,
            dim=256,
            num_layers=8,
            dropout_rate=dropout_rate
        )

        self.b2 = DilatedConv(
            in_c=1024,
            out_c=256,
            dropout_rate=dropout_rate
        )

        self.d1 = DecoderBlock(
            in_c=[512, 512],
            out_c=256,
            dropout_rate=dropout_rate
        )

        self.d2 = DecoderBlock(
            in_c=[256, 256],
            out_c=128,
            dropout_rate=dropout_rate
        )

        self.d3 = DecoderBlock(
            in_c=[128, 64],
            out_c=64,
            dropout_rate=dropout_rate
        )

        self.d4 = DecoderBlock(
            in_c=[64, 3],
            out_c=32,
            dropout_rate=dropout_rate
        )

        self.output = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

    def forward(self, x):
        s0 = x

        s1 = self.layer0(s0)
        s2 = self.layer1(s1)
        s3 = self.layer2(s2)
        s4 = self.layer3(s3)

        b1 = self.b1(s4)
        b2 = self.b2(s4)

        b3 = torch.cat([b1, b2], dim=1)

        d1 = self.d1(b3, s3)
        d2 = self.d2(d1, s2)
        d3 = self.d3(d2, s1)
        d4 = self.d4(d3, s0)

        return self.output(d4)

# =========================================================
# 2. 权重读取
# =========================================================
def torch_load_compat(path: Path, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def strip_prefix_if_present(
    state_dict: Dict[str, torch.Tensor],
    prefix: str,
) -> Dict[str, torch.Tensor]:
    if state_dict and all(key.startswith(prefix) for key in state_dict):
        return {key[len(prefix):]: value for key, value in state_dict.items()}
    return state_dict


def extract_state_dict(checkpoint) -> Dict[str, torch.Tensor]:
    candidate_keys = [
        "student_state_dict",
        "model_state_dict",
        "state_dict",
        "model",
    ]

    state_dict = None

    for key in candidate_keys:
        value = checkpoint.get(key)
        if (
            isinstance(value, dict)
            and value
            and all(torch.is_tensor(v) for v in value.values())
        ):
            state_dict = value
            break

    state_dict = strip_prefix_if_present(state_dict, "module.")
    state_dict = strip_prefix_if_present(state_dict, "model.")

    return state_dict

def find_numeric_value(checkpoint: dict, keys: Iterable[str]) -> Optional[float]:
    for key in keys:
        value = checkpoint.get(key)
        if isinstance(value, (int, float, np.integer, np.floating)):
            return float(value)

    for container_key in ["config", "meta", "metadata", "normalization", "preprocess"]:
        nested = checkpoint.get(container_key)
        if not isinstance(nested, dict):
            continue
        for key in keys:
            value = nested.get(key)
            if isinstance(value, (int, float, np.integer, np.floating)):
                return float(value)

    return None


def load_model_and_checkpoint(checkpoint_path: Path):
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"权重文件不存在: {checkpoint_path}")

    checkpoint = torch_load_compat(checkpoint_path, DEVICE)
    state_dict = extract_state_dict(checkpoint)

    model = RCSDT_UNet(
        dropout_rate=DROPOUT_RATE,
        pretrained=False,
    ).to(DEVICE)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            "权重与当前 RCSDT-UNet 网络结构不匹配。"
            "请确认 best_model.pth 就是该网络保存的权重。\n"
            f"原始错误：{exc}"
        ) from exc

    model.eval()
    print("权重加载成功")
    return model, checkpoint


# =========================================================
# 3. 输入图像与归一化
# =========================================================
def is_supported_image(path: Path) -> bool:
    if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        return False

    if SKIP_MASK_NAMED_FILES and "mask" in path.stem.lower():
        return False

    return True


def collect_images(
    input_dir: Path,
    patient_folders: List[str],
    recursive: bool,
) -> List[Path]:
    """
    只读取 data 下指定的三个病人文件夹：

        data/1
        data/1 (32)
        data/1 (39)

    返回的路径仍保留病人文件夹层级，因此预测 mask 会分别保存到：

        mask/1
        mask/1 (32)
        mask/1 (39)
    """
    if not input_dir.is_dir():
        raise FileNotFoundError(f"输入根目录不存在: {input_dir}")

    images = []
    missing_folders = []

    for patient_folder_name in patient_folders:
        patient_dir = input_dir / patient_folder_name

        if not patient_dir.is_dir():
            missing_folders.append(str(patient_dir))
            continue

        iterator = (
            patient_dir.rglob("*")
            if recursive
            else patient_dir.glob("*")
        )

        patient_images = sorted(
            path
            for path in iterator
            if is_supported_image(path)
        )

        print(
            f"病人文件夹 {patient_folder_name}: "
            f"检测到 {len(patient_images)} 张 JPG"
        )

        images.extend(patient_images)

    if missing_folders:
        print("\n以下病人文件夹不存在，将跳过：")
        for folder in missing_folders:
            print(f"  {folder}")

    if not images:
        raise ValueError(
            "三个病人文件夹中没有找到可处理的 JPG 图片。"
        )

    return sorted(images)

def calculate_data_mean_std(image_paths: List[Path]) -> Tuple[float, float]:
    pixel_sum = 0.0
    pixel_square_sum = 0.0
    pixel_count = 0
    valid_count = 0

    for image_path in image_paths:
        try:
            with Image.open(image_path) as source:
                gray = source.convert("L").resize(IMAGE_SIZE, BICUBIC)
            array = np.asarray(gray, dtype=np.float64) / 255.0
        except Exception as exc:
            print(f"统计归一化参数时跳过损坏图片: {image_path}，原因: {exc}")
            continue

        pixel_sum += float(array.sum())
        pixel_square_sum += float(np.square(array).sum())
        pixel_count += int(array.size)
        valid_count += 1

    if valid_count == 0 or pixel_count == 0:
        raise ValueError("没有可用于计算 mean/std 的有效图片。")

    mean = pixel_sum / pixel_count
    variance = max(pixel_square_sum / pixel_count - mean * mean, 0.0)
    std = max(float(np.sqrt(variance)), 1e-5)

    print(f"根据输入文件夹 {valid_count} 张图片计算归一化参数")
    print(f"data mean={mean:.8f}, std={std:.8f}")
    return float(mean), float(std)


def resolve_normalization(checkpoint, image_paths: List[Path]) -> Tuple[float, float]:
    return float(TRAIN_MEAN), float(TRAIN_STD)

def preprocess_image(
    image_path: Path,
    mean: float,
    std: float,
) -> Tuple[torch.Tensor, Tuple[int, int]]:
    with Image.open(image_path) as source:
        original_size = source.size
        gray = source.convert("L")
        resized = gray.resize(IMAGE_SIZE, BICUBIC)

    image_np = np.asarray(resized, dtype=np.float32) / 255.0
    image_np = (image_np - mean) / max(std, 1e-5)

    tensor = torch.from_numpy(image_np).unsqueeze(0).unsqueeze(0)
    tensor = tensor.repeat(1, 3, 1, 1).to(DEVICE, non_blocking=True)
    return tensor, original_size


# =========================================================
# 4. 文件夹批量分割
# =========================================================
def build_output_path(
    image_path: Path,
    input_dir: Path,
    output_dir: Path,
) -> Path:
    relative_parent = image_path.relative_to(input_dir).parent
    return output_dir / relative_parent / f"{image_path.stem}_mask.png"


def predict_one_image(
    model: RCSDT_UNet,
    image_path: Path,
    output_path: Path,
    mean: float,
    std: float,
    threshold: float,
) -> None:
    image_tensor, original_size = preprocess_image(image_path, mean, std)

    with torch.inference_mode():
        logits = model(image_tensor)
        probability = torch.sigmoid(logits)[0, 0].cpu().numpy()

    binary_mask = (probability > threshold).astype(np.uint8) * 255
    mask_image = Image.fromarray(binary_mask, mode="L")

    if RESTORE_ORIGINAL_SIZE and mask_image.size != original_size:
        mask_image = mask_image.resize(original_size, NEAREST)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    mask_image.save(output_path, format="PNG")

def segment_folder(
    model: RCSDT_UNet,
    image_paths: List[Path],
    input_dir: Path,
    output_dir: Path,
    mean: float,
    std: float,
    threshold: float,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failed_count = 0

    print(f"\n共检测到 {len(image_paths)} 张待分割图片")
    print(f"mask 输出目录: {output_dir}")

    for index, image_path in enumerate(image_paths, start=1):
        output_path = build_output_path(image_path, input_dir, output_dir)

        try:
            predict_one_image(
                model=model,
                image_path=image_path,
                output_path=output_path,
                mean=mean,
                std=std,
                threshold=threshold,
            )
            success_count += 1
            print(
                f"[{index}/{len(image_paths)}] 完成: "
                f"{image_path.name} -> {output_path.name}"
            )
        except Exception as exc:
            failed_count += 1
            print(f"[{index}/{len(image_paths)}] 失败: {image_path} | {exc}")

    print("\n文件夹分割完成")
    print(f"成功: {success_count}")
    print(f"失败: {failed_count}")
    print(f"结果目录: {output_dir.resolve()}")


def main() -> None:
    input_dir = Path(INPUT_DIR)
    output_dir = Path(OUTPUT_MASK_DIR)
    checkpoint_path = Path(CHECKPOINT_PATH)

    # 自动创建三个病人对应的 mask 输出文件夹。
    for patient_folder_name in PATIENT_FOLDERS:
        (output_dir / patient_folder_name).mkdir(
            parents=True,
            exist_ok=True,
        )

    if not 0.0 <= THRESHOLD <= 1.0:
        raise ValueError(f"THRESHOLD 必须位于 [0, 1]，当前为 {THRESHOLD}")

    image_paths = collect_images(
        input_dir=input_dir,
        patient_folders=PATIENT_FOLDERS,
        recursive=RECURSIVE,
    )

    model, checkpoint = load_model_and_checkpoint(checkpoint_path)
    mean, std = resolve_normalization(checkpoint, image_paths)

    segment_folder(
        model=model,
        image_paths=image_paths,
        input_dir=input_dir,
        output_dir=output_dir,
        mean=mean,
        std=std,
        threshold=THRESHOLD,
    )


main()
